In [10]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

In [11]:
BASE = Path("../data/processed/primary")

INPUT_FILE = BASE / "inpatient_ml_preprocessed.csv"
OUTPUT_FILE = BASE / "inpatient_anomaly_scores.csv"


In [12]:

CHUNK_SIZE = 250_000
TRAIN_SAMPLE_SIZE = 50_000
RANDOM_STATE = 42



In [13]:
# IDENTIFY FEATURES

header = pd.read_csv(INPUT_FILE, nrows=0)

ID_COLS = [
    "CLM_ID",
    "DESYNPUF_ID"
]

FEATURE_COLS = [
    c for c in header.columns
    if c not in ID_COLS
]

print("=" * 80)
print("INPATIENT ANOMALY DETECTION")
print("=" * 80)

print("Total columns:", len(header.columns))
print("Feature count:", len(FEATURE_COLS))
print("Training sample:", TRAIN_SAMPLE_SIZE)
print("Chunk size:", CHUNK_SIZE)


INPATIENT ANOMALY DETECTION
Total columns: 62
Feature count: 62
Training sample: 50000
Chunk size: 250000


In [14]:

# CREATE TRAINING SAMPLE

print("\nCreating training sample...")

rng = np.random.default_rng(RANDOM_STATE)

sample_parts = []
collected = 0

for chunk in pd.read_csv(
    INPUT_FILE,
    usecols=FEATURE_COLS,
    chunksize=CHUNK_SIZE,
    low_memory=False
):

    remaining = TRAIN_SAMPLE_SIZE - collected

    if remaining <= 0:
        break

    if len(chunk) <= remaining:
        sample_parts.append(chunk)
        collected += len(chunk)
    else:
        indices = rng.choice(
            len(chunk),
            size=remaining,
            replace=False
        )

        sample_parts.append(
            chunk.iloc[indices]
        )

        collected += remaining

    print(
        f"Collected {collected:,} / "
        f"{TRAIN_SAMPLE_SIZE:,}"
    )

training_sample = pd.concat(
    sample_parts,
    ignore_index=True
)

print("\nTraining sample shape:")
print(training_sample.shape)




Creating training sample...
Collected 50,000 / 50,000

Training sample shape:
(50000, 62)


In [15]:
# CONVERT TO NUMERIC

for col in training_sample.columns:

    training_sample[col] = pd.to_numeric(
        training_sample[col],
        errors="coerce"
    )

# Replace missing/infinite values

training_sample = training_sample.replace(
    [np.inf, -np.inf],
    np.nan
)

# Median imputation

for col in training_sample.columns:

    if training_sample[col].isna().any():

        median_value = training_sample[col].median()

        if pd.isna(median_value):
            median_value = 0

        training_sample[col] = (
            training_sample[col]
            .fillna(median_value)
        )


In [16]:

# SCALE

print("Scaling features...")

scaler = StandardScaler()

X_train = scaler.fit_transform(
    training_sample
)

print("Scaled training matrix:")
print(X_train.shape)

del training_sample
del sample_parts


Scaling features...
Scaled training matrix:
(50000, 62)


In [17]:

# TRAIN ISOLATION FOREST

print("\nTraining Isolation Forest...")

model = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

model.fit(X_train)

del X_train

print("Model trained.")



Training Isolation Forest...
Model trained.


In [19]:
# ============================================================
# MEMORY-SAFE INPATIENT SCORING
# ============================================================

ML_FILE = BASE / "inpatient_ml_preprocessed.csv"
ID_FILE = BASE / "inpatient_claims_fully_enriched.csv"

print("\n" + "=" * 80)
print("SCORING ALL INPATIENT CLAIMS")
print("=" * 80)

# ------------------------------------------------------------
# Verify row counts before scoring
# ------------------------------------------------------------

ml_rows = sum(
    1 for _ in open(ML_FILE, "r", encoding="utf-8")
) - 1

id_rows = sum(
    1 for _ in open(ID_FILE, "r", encoding="utf-8")
) - 1

print("ML rows:", f"{ml_rows:,}")
print("ID rows:", f"{id_rows:,}")

if ml_rows != id_rows:
    raise ValueError(
        f"Row mismatch! ML={ml_rows:,}, IDs={id_rows:,}"
    )

print("Row counts match.")

# ------------------------------------------------------------
# Get ID columns from original enriched inpatient data
# ------------------------------------------------------------

ID_COLS = [
    "CLM_ID",
    "DESYNPUF_ID"
]

# Check source columns

id_header = pd.read_csv(
    ID_FILE,
    nrows=0
)

missing_ids = [
    c for c in ID_COLS
    if c not in id_header.columns
]

if missing_ids:
    raise ValueError(
        f"Missing ID columns in enriched inpatient file: "
        f"{missing_ids}"
    )

# ------------------------------------------------------------
# Score files in parallel chunks
# ------------------------------------------------------------

if OUTPUT_FILE.exists():
    OUTPUT_FILE.unlink()

ml_reader = pd.read_csv(
    ML_FILE,
    chunksize=CHUNK_SIZE,
    low_memory=False
)

id_reader = pd.read_csv(
    ID_FILE,
    usecols=ID_COLS,
    chunksize=CHUNK_SIZE,
    low_memory=False
)

first_chunk = True
total_scored = 0

for chunk_number, (X, ids) in enumerate(
    zip(ml_reader, id_reader),
    start=1
):

    # --------------------------------------------------------
    # Safety check
    # --------------------------------------------------------

    if len(X) != len(ids):
        raise ValueError(
            f"Chunk {chunk_number} row mismatch: "
            f"ML={len(X)}, IDs={len(ids)}"
        )

    # --------------------------------------------------------
    # Convert ML features to numeric
    # --------------------------------------------------------

    X = X.apply(
        pd.to_numeric,
        errors="coerce"
    )

    X = X.replace(
        [np.inf, -np.inf],
        np.nan
    )

    # --------------------------------------------------------
    # Impute missing values
    # --------------------------------------------------------

    for col in X.columns:

        if X[col].isna().any():

            median_value = X[col].median()

            if pd.isna(median_value):
                median_value = 0

            X[col] = X[col].fillna(
                median_value
            )

    # --------------------------------------------------------
    # Scale
    # --------------------------------------------------------

    X_scaled = scaler.transform(X)

    # --------------------------------------------------------
    # Generate anomaly score
    #
    # Higher = more anomalous
    # --------------------------------------------------------

    scores = -model.decision_function(
        X_scaled
    )

    # --------------------------------------------------------
    # Build output
    # --------------------------------------------------------

    result = ids.copy()

    result["ANOMALY_SCORE"] = scores

    result.to_csv(
        OUTPUT_FILE,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

    total_scored += len(X)

    print(
        f"Scored {total_scored:,} claims..."
    )

# ============================================================
# FINAL CHECK
# ============================================================

print("\n" + "=" * 80)
print("INPATIENT MODEL COMPLETE")
print("=" * 80)

print(
    "Claims scored:",
    f"{total_scored:,}"
)

print(
    "Expected:",
    f"{ml_rows:,}"
)

print(
    "Score file:",
    OUTPUT_FILE
)


SCORING ALL INPATIENT CLAIMS
ML rows: 66,773
ID rows: 66,773
Row counts match.
Scored 66,773 claims...

INPATIENT MODEL COMPLETE
Claims scored: 66,773
Expected: 66,773
Score file: ..\data\processed\primary\inpatient_anomaly_scores.csv


In [20]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("../data/processed/primary")

SCORE_FILE = BASE / "inpatient_anomaly_scores.csv"

scores = pd.read_csv(SCORE_FILE)

print("=" * 90)
print("INPATIENT ANOMALY SCORE ANALYSIS")
print("=" * 90)

print("\nScore rows:", len(scores))
print("Score columns:", list(scores.columns))

print("\nMissing scores:", scores["ANOMALY_SCORE"].isna().sum())
print("Infinite scores:", np.isinf(scores["ANOMALY_SCORE"]).sum())
print("Unique scores:", scores["ANOMALY_SCORE"].nunique())

print("\n" + "=" * 90)
print("ANOMALY SCORE DISTRIBUTION")
print("=" * 90)

print(
    scores["ANOMALY_SCORE"].describe(
        percentiles=[
            0.001,
            0.005,
            0.01,
            0.02,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.995,
            0.999
        ]
    )
)

print("\n" + "=" * 90)
print("HIGH ANOMALY THRESHOLDS")
print("=" * 90)

for pct in [0.10, 0.05, 0.03, 0.02, 0.01, 0.005, 0.001]:

    threshold = scores["ANOMALY_SCORE"].quantile(
        1 - pct
    )

    count = (
        scores["ANOMALY_SCORE"] >= threshold
    ).sum()

    print(
        f"Top {pct*100:.1f}% | "
        f"threshold={threshold:.6f} | "
        f"claims={count:,}"
    )

print("\n" + "=" * 90)
print("SCORE RANGE")
print("=" * 90)

print(
    "Minimum:",
    scores["ANOMALY_SCORE"].min()
)

print(
    "Maximum:",
    scores["ANOMALY_SCORE"].max()
)

print(
    "Mean:",
    scores["ANOMALY_SCORE"].mean()
)

print(
    "Median:",
    scores["ANOMALY_SCORE"].median()
)

print("\n" + "=" * 90)
print("TOP 20 ANOMALY SCORES")
print("=" * 90)

print(
    scores
    .sort_values(
        "ANOMALY_SCORE",
        ascending=False
    )
    .head(20)
    .to_string(index=False)
)

INPATIENT ANOMALY SCORE ANALYSIS

Score rows: 66773
Score columns: ['DESYNPUF_ID', 'CLM_ID', 'ANOMALY_SCORE']

Missing scores: 0
Infinite scores: 0
Unique scores: 66737

ANOMALY SCORE DISTRIBUTION
count    66773.000000
mean        -0.064915
std          0.032177
min         -0.131779
0.1%        -0.124514
0.5%        -0.119330
1%          -0.116404
2%          -0.113055
5%          -0.106765
10%         -0.100442
25%         -0.087897
50%         -0.070688
75%         -0.048226
90%         -0.020857
95%         -0.002016
99%          0.033120
99.5%        0.045899
99.9%        0.074711
max          0.123256
Name: ANOMALY_SCORE, dtype: float64

HIGH ANOMALY THRESHOLDS
Top 10.0% | threshold=-0.020857 | claims=6,678
Top 5.0% | threshold=-0.002016 | claims=3,339
Top 3.0% | threshold=0.010040 | claims=2,004
Top 2.0% | threshold=0.018832 | claims=1,336
Top 1.0% | threshold=0.033120 | claims=668
Top 0.5% | threshold=0.045899 | claims=334
Top 0.1% | threshold=0.074711 | claims=67

SCORE RANGE


In [21]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("../data/processed/primary")

SCORE_FILE = BASE / "inpatient_anomaly_scores.csv"
DATA_FILE = BASE / "inpatient_claims_fully_enriched.csv"
OUTPUT_FILE = BASE / "inpatient_top_anomalies.csv"

TOP_N = 100
CHUNK_SIZE = 250_000

print("=" * 80)
print("RETRIEVING TOP INPATIENT ANOMALIES")
print("=" * 80)

# ============================================================
# LOAD SCORES
# ============================================================

scores = pd.read_csv(SCORE_FILE)

scores["ROW_ID"] = np.arange(len(scores))

top_scores = (
    scores
    .sort_values(
        "ANOMALY_SCORE",
        ascending=False
    )
    .head(TOP_N)
    .copy()
)

print("Scores:", f"{len(scores):,}")
print("Top anomalies:", len(top_scores))

# Map score row position -> rank

top_scores["ANOMALY_RANK"] = range(
    1,
    len(top_scores) + 1
)

score_lookup = {
    row_id: (rank, score)
    for row_id, rank, score
    in zip(
        top_scores["ROW_ID"],
        top_scores["ANOMALY_RANK"],
        top_scores["ANOMALY_SCORE"]
    )
}

# ============================================================
# COLUMNS TO RETRIEVE
# ============================================================

DETAIL_COLS = [
    "DESYNPUF_ID",
    "CLM_ID",
    "PRVDR_NUM",
    "CLM_FROM_DT",
    "CLM_THRU_DT",
    "CLM_ADMSN_DT",
    "CLM_UTLZTN_DAY_CNT",
    "CLM_DRG_CD",
    "CLM_PMT_AMT",
    "NCH_PRMRY_PYR_CLM_PD_AMT",
    "CLAIM_DURATION_DAYS",
    "DIAGNOSIS_COUNT",
    "PROCEDURE_COUNT",
    "HAS_NEGATIVE_PAYMENT",
    "HAS_PRIMARY_PAYER_PAYMENT",
    "CLAIM_YEAR",
    "AT_PHYSN_NPI",
    "OP_PHYSN_NPI",
    "OT_PHYSN_NPI"
]

# Check columns

header = pd.read_csv(
    DATA_FILE,
    nrows=0
)

available = set(header.columns)

DETAIL_COLS = [
    c for c in DETAIL_COLS
    if c in available
]

print("\nColumns being retrieved:")
print(DETAIL_COLS)

# ============================================================
# RETRIEVE TOP CLAIMS
# ============================================================

matches = []
current_row = 0

for chunk_number, chunk in enumerate(
    pd.read_csv(
        DATA_FILE,
        usecols=DETAIL_COLS,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    start = current_row
    end = current_row + len(chunk)

    relevant_ids = [
        row_id
        for row_id in score_lookup
        if start <= row_id < end
    ]

    if relevant_ids:

        for row_id in relevant_ids:

            local_index = row_id - start

            row = chunk.iloc[
                local_index
            ].to_dict()

            rank, score = score_lookup[row_id]

            row["ANOMALY_RANK"] = rank
            row["ANOMALY_SCORE"] = score

            matches.append(row)

    current_row = end

    print(
        f"Chunk {chunk_number}: "
        f"{current_row:,} rows processed"
    )

# ============================================================
# FINAL TABLE
# ============================================================

top_anomalies = pd.DataFrame(matches)

top_anomalies = (
    top_anomalies
    .sort_values(
        "ANOMALY_RANK"
    )
    .reset_index(drop=True)
)

top_anomalies.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 80)
print("TOP INPATIENT ANOMALIES SAVED")
print("=" * 80)

print("Rows:", len(top_anomalies))
print("Saved:", OUTPUT_FILE)

print("\nTOP 20:")

print(
    top_anomalies
    .head(20)
    .to_string(index=False)
)

RETRIEVING TOP INPATIENT ANOMALIES
Scores: 66,773
Top anomalies: 100

Columns being retrieved:
['DESYNPUF_ID', 'CLM_ID', 'PRVDR_NUM', 'CLM_FROM_DT', 'CLM_THRU_DT', 'CLM_ADMSN_DT', 'CLM_UTLZTN_DAY_CNT', 'CLM_DRG_CD', 'CLM_PMT_AMT', 'NCH_PRMRY_PYR_CLM_PD_AMT', 'CLAIM_DURATION_DAYS', 'DIAGNOSIS_COUNT', 'PROCEDURE_COUNT', 'HAS_NEGATIVE_PAYMENT', 'HAS_PRIMARY_PAYER_PAYMENT', 'CLAIM_YEAR', 'AT_PHYSN_NPI', 'OP_PHYSN_NPI', 'OT_PHYSN_NPI']
Chunk 1: 66,773 rows processed

TOP INPATIENT ANOMALIES SAVED
Rows: 100
Saved: ..\data\processed\primary\inpatient_top_anomalies.csv

TOP 20:
     DESYNPUF_ID          CLM_ID CLM_FROM_DT CLM_THRU_DT PRVDR_NUM  CLM_PMT_AMT  NCH_PRMRY_PYR_CLM_PD_AMT  AT_PHYSN_NPI  OP_PHYSN_NPI  OT_PHYSN_NPI CLM_ADMSN_DT  CLM_UTLZTN_DAY_CNT CLM_DRG_CD  CLAIM_DURATION_DAYS  DIAGNOSIS_COUNT  PROCEDURE_COUNT  HAS_NEGATIVE_PAYMENT  HAS_PRIMARY_PAYER_PAYMENT  CLAIM_YEAR  ANOMALY_RANK  ANOMALY_SCORE
CED6AE2B33D1F957 196071177024706  2009-08-25  2009-09-04    4413WV      31000.0       

In [22]:
print("=" * 80)
print("CHECKING TRAINED ANOMALY MODELS")
print("=" * 80)

for name in [
    "carrier_model",
    "outpatient_model",
    "inpatient_model",
    "carrier_if",
    "outpatient_if",
    "inpatient_if",
    "carrier_scaler",
    "outpatient_scaler",
    "inpatient_scaler",
    "scaler",
]:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: {type(obj).__name__}")

print("=" * 80)

CHECKING TRAINED ANOMALY MODELS
scaler: StandardScaler


In [24]:
from sklearn.ensemble import IsolationForest

print("=" * 80)
print("TRAINED ISOLATION FOREST OBJECTS")
print("=" * 80)

for name, obj in globals().items():
    try:
        if isinstance(obj, IsolationForest):
            print(f"{name} -> IsolationForest")
    except:
        pass

print("\n" + "=" * 80)
print("SCALERS")
print("=" * 80)

from sklearn.preprocessing import RobustScaler, StandardScaler

for name, obj in globals().items():
    try:
        if isinstance(obj, (RobustScaler, StandardScaler)):
            print(f"{name} -> {type(obj).__name__}")
    except:
        pass

TRAINED ISOLATION FOREST OBJECTS
model -> IsolationForest

SCALERS
scaler -> StandardScaler


In [25]:
import joblib
import json
from pathlib import Path

# ============================================================
# SAVE INPATIENT ANOMALY MODEL
# ============================================================

MODEL_DIR = Path("../models/anomaly/inpatient")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    model,
    MODEL_DIR / "isolation_forest.joblib"
)

joblib.dump(
    scaler,
    MODEL_DIR / "scaler.joblib"
)

feature_schema = {
    "claim_type": "INPATIENT",
    "model_type": "IsolationForest",
    "scaler_type": type(scaler).__name__,
    "feature_count": len(FEATURE_COLS),
    "features": list(FEATURE_COLS)
}

with open(
    MODEL_DIR / "feature_schema.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(feature_schema, f, indent=2)

print("=" * 80)
print("INPATIENT MODEL SAVED")
print("=" * 80)

print("Model:", MODEL_DIR / "isolation_forest.joblib")
print("Scaler:", MODEL_DIR / "scaler.joblib")
print("Schema:", MODEL_DIR / "feature_schema.json")
print("Features:", len(FEATURE_COLS))
print("Scaler type:", type(scaler).__name__)

INPATIENT MODEL SAVED
Model: ..\models\anomaly\inpatient\isolation_forest.joblib
Scaler: ..\models\anomaly\inpatient\scaler.joblib
Schema: ..\models\anomaly\inpatient\feature_schema.json
Features: 62
Scaler type: StandardScaler


In [26]:
from pathlib import Path
import joblib
import json

BASE = Path("../models/anomaly")

print("=" * 80)
print("ANOMALY MODEL CHECKPOINT")
print("=" * 80)

for claim_type in ["carrier", "outpatient", "inpatient"]:

    folder = BASE / claim_type

    model_file = folder / "isolation_forest.joblib"
    scaler_file = folder / "scaler.joblib"
    schema_file = folder / "feature_schema.json"

    print(f"\n{claim_type.upper()}")

    print("Model exists:", model_file.exists())
    print("Scaler exists:", scaler_file.exists())
    print("Schema exists:", schema_file.exists())

    if model_file.exists():
        loaded_model = joblib.load(model_file)
        print("Loaded model:", type(loaded_model).__name__)

    if scaler_file.exists():
        loaded_scaler = joblib.load(scaler_file)
        print("Loaded scaler:", type(loaded_scaler).__name__)

    if schema_file.exists():
        with open(schema_file, "r", encoding="utf-8") as f:
            schema = json.load(f)

        print("Claim type:", schema["claim_type"])
        print("Feature count:", schema["feature_count"])

print("\n" + "=" * 80)
print("CHECKPOINT COMPLETE")
print("=" * 80)

ANOMALY MODEL CHECKPOINT

CARRIER
Model exists: True
Scaler exists: True
Schema exists: True
Loaded model: IsolationForest
Loaded scaler: RobustScaler
Claim type: CARRIER
Feature count: 53

OUTPATIENT
Model exists: True
Scaler exists: True
Schema exists: True
Loaded model: IsolationForest
Loaded scaler: RobustScaler
Claim type: OUTPATIENT
Feature count: 51

INPATIENT
Model exists: True
Scaler exists: True
Schema exists: True
Loaded model: IsolationForest
Loaded scaler: StandardScaler
Claim type: INPATIENT
Feature count: 62

CHECKPOINT COMPLETE
